# EDA & Regression Analysis

## Important data note
The uploaded CSV contains `Duration`, `Date`, `Pulse`, `Maxpulse`, and `Calories`. It does **not** contain a Sales field. Therefore, a sales-prediction model cannot be validly trained from this dataset. This notebook performs the requested regression workflow using **Calories as the available numeric outcome**, and explicitly documents the limitation.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
df = pd.read_csv('../data/data.csv')
df.info()
df.head()


## 1. Exploratory Data Analysis


In [ ]:
print(df.describe())
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicates:', df.duplicated().sum())


In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
print(df.corr(numeric_only=True).round(3))
df[['Duration','Pulse','Maxpulse','Calories']].hist(figsize=(10,7))
plt.tight_layout(); plt.show()


## 2. Regression Model
Because Sales is absent, Calories is used as the outcome variable. Date is converted to an ordinal numeric feature and missing values are imputed with medians.


In [ ]:
model_df = df.copy()
model_df['Date_ordinal'] = model_df['Date'].map(lambda x: x.toordinal() if pd.notna(x) else np.nan)
model_df = model_df.drop(columns=['Date']).dropna(subset=['Calories'])
X = model_df.drop(columns=['Calories']); y = model_df['Calories']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
pipe = Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('regressor',LinearRegression())])
pipe.fit(X_train,y_train)
pred = pipe.predict(X_test)
print('MAE:', mean_absolute_error(y_test,pred))
print('RMSE:', mean_squared_error(y_test,pred)**0.5)
print('R2:', r2_score(y_test,pred))


## 3. Power BI Dashboard Plan
Load `outputs/PowerBI_Data.xlsx` (or the cleaned CSV) into Power BI. Recommended visuals: KPI cards for record count, average Calories, average Duration, and average Pulse; scatter plot of Duration vs Calories; line chart of Calories by Date; and a correlation/metric table.


## 4. Findings & Recommendations
- The dataset has 32 records.
- Date has 1 missing value and Calories has 2 missing values.
- The available dataset is an exercise/health-style dataset rather than a sales dataset.
- A sales model should not be claimed until a Sales/Revenue target is supplied.
- For the available outcome, the regression provides a baseline for predicting Calories from the other fields.


## 5. Limitation
A `.pbix` Power BI Desktop file cannot be generated reliably outside Power BI Desktop. The project includes Power BI-ready data and a dashboard specification so the PBIX can be created quickly in Power BI Desktop.
